# Backbone benchmarking -- Kaggle runner

Thin wrapper. All logic lives in the `bbeval` package; this notebook only
installs it, points it at the mounted datasets, and runs it.

Attach **MVTec AD** and **VisA** as datasets and enable a **GPU** accelerator.
The run finishes by writing a single ZIP of every artefact, next to
`output_root`, ready to download from the Kaggle output panel.


## 1. Install

In [ ]:
%pip install -q -e /kaggle/working/backbone-eval[siglip2]
# CLIP is a git dependency; skip this line for a SigLIP2-only run.
%pip install -q "git+https://github.com/openai/CLIP.git"


## 2. Settings

`siglip2_dense_readout` is the variable under test:

* `"map_token"` -- pool each patch through SigLIP2's own attention-pooling head
* `"raw"` -- leave trunk tokens unprojected; the CLIP-shaped control that
  reproduces the published chance-level localisation

`corruptions_enabled=False` keeps this to the clean backbone comparison. Turning
it on multiplies the sweep by 34 settings per category and will not fit in one
Kaggle session -- see the README.


In [ ]:
from bbeval import BackboneEvalConfig, run_evaluation

config = BackboneEvalConfig(
    mvtec_root="/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection",
    visa_root="/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922",
    output_root="/kaggle/working/results",
    weights_dir="/kaggle/working/weights",
    backbones=("clip", "siglip2"),
    siglip2_dense_readout="map_token",
    corruptions_enabled=False,
    num_workers=2,
    device="cuda",
)
print("config id:", config.fingerprint())


## 3. Smoke test

Two categories, four images each. `limit` suppresses saving, so this cannot
pollute the real artefacts -- it only proves the weights load and the shapes
line up before committing hours of GPU time.


In [ ]:
from dataclasses import replace
from bbeval.engine import load_backbones, run_shard, sweep_plan
from bbeval.prompts import build_fixed_text

smoke = replace(config, limit=4, archive_results=False,
                categories={"mvtec": ("hazelnut",), "visa": ("candle",)})
backbones = load_backbones(smoke)

for name, backbone in backbones.items():
    text = build_fixed_text(smoke, backbone, "hazelnut")
    outputs, masks = run_shard(smoke, backbone, {"fixed": text}, "mvtec",
                               "hazelnut", "clean", 0, save=False)
    scores, maps = outputs["fixed"]
    print(f"{name:<10} scores {tuple(scores.shape)} maps {tuple(maps.shape)} "
          f"range [{maps.min():.3f}, {maps.max():.3f}]")


## 4. Run

Fits one prompt set per source dataset, scores every category of the *other*
dataset, and writes low-resolution anomaly maps, raw scores, ground truth, the
run manifest and the metric tables.

Set `resume=True` (the default) so an interrupted session picks up where it
stopped rather than recomputing.


In [ ]:
result = run_evaluation(config)
result


## 5. Download

`result["archive"]` is the single ZIP containing artefacts, prompt checkpoints,
tables and the run manifest. It appears in the Kaggle output panel.


In [ ]:
import os
print(result["archive"], f'{os.path.getsize(result["archive"]) / 1e6:.0f} MB')
